# Chapter 7: Sets

A **set** stores distinct values: adding an existing member does not create a duplicate. Mathematical sets have no positional order. JavaScript's native `Set`, however, preserves insertion order when iterating.

This notebook covers an object-backed `MySet`, union, intersection, difference, subsets and supersets, practical applications, native `Set`, and removing duplicates from a sorted array.

All implementations and examples are in executable JavaScript cells. The draft's TypeScript annotations and file imports have been removed to match the other chapter notebooks. Select **JavaScript (Node.js)** and **Run → Run All Cells**. Run the class and function definitions before their examples. Cells can be rerun without declaration conflicts.

## First-time setup: install Jupyter and the JavaScript kernel

This `.ipynb` file opens in Jupyter. **Python** runs Jupyter; **Node.js** runs the JavaScript examples through the **IJavascript kernel**. Install [Python 3](https://www.python.org/downloads/) and [Node.js LTS](https://nodejs.org/en/download/) first if needed.

Run these commands in a **terminal**, not a notebook code cell. Start in the project's root folder.

**macOS / Linux:**

```sh
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
python -m pip install jupyterlab notebook
npm install -g ijavascript
ijsinstall --spec-path=full --hide-undefined
jupyter kernelspec list
jupyter lab src/chapter7/chapter7_examples.ipynb
```

If `.venv` already exists and works, skip creating it. On **Windows Command Prompt**, replace the first two commands with `py -m venv .venv` and `.venv\Scripts\activate.bat`; the remaining commands are the same.

In Jupyter, select **JavaScript (Node.js)** and choose **Run → Run All Cells**. If `jsbook` is reported missing, select the installed JavaScript kernel through **Kernel → Change Kernel** and save. `jsbook` is only the original author's local kernel name. The final check cell should print `All Chapter 7 checks passed.`

For the Jupyter Notebook interface, launch with `jupyter notebook src/chapter7/chapter7_examples.ipynb` instead. Later, just activate `.venv` and launch Jupyter again; no reinstallation is needed.

See the [project setup guide](../../README.md) for troubleshooting. Installation references: [Jupyter](https://jupyter.org/install), [IJavascript](https://github.com/n-riesco/ijavascript/blob/master/doc/install.md), and [kernel options](https://github.com/n-riesco/ijavascript/blob/master/doc/usage.md).

## 1. Distinct values and the empty set

The empty set, written ∅ or `{}`, has no members. For example, there are no primes strictly between 24 and 29 (29 itself is excluded). Sets are useful for unique tags, membership checks, and combining collections.

In [1]:
(() => {
  const tags = new Set(['programming', 'algorithms', 'programming']);
  console.log('Unique tags:', [...tags]); // ['programming', 'algorithms']
  console.log('Number of tags:', tags.size); // 2
  console.log('Empty set size:', new Set().size); // 0
})();

Unique tags: [ 'programming', 'algorithms' ]


Number of tags: 2


Empty set size: 0


## 2. Creating MySet from scratch

Our teaching implementation stores **strings only**, matching the draft's typed `add(value: string)` API. A null-prototype object (`Object.create(null)`) stores each string as a key with the value `true`. `Object.hasOwn()` checks membership without inherited properties interfering. A normal `{}` inherits names such as `toString`, so using `in` on it is unsafe for this purpose.

Rejecting non-strings prevents accidental collisions between `1` and `'1'`, or unrelated objects converted to the same property name. Use native `Set` when you need numbers, objects, or mixed types.

`add()` and `delete()` return booleans; `size` is a getter. `addAll()` fills in the helper used throughout the draft. The constructor accepts an iterable for convenient examples. The class includes both the simple and optimized intersection implementations for comparison. All operations that produce sets create a new result without changing either input.

In [2]:
var MySet = class MySet {
  #items = Object.create(null);
  #size = 0;

  constructor(values = []) {
    this.addAll(values);
  }

  #checkValue(value) {
    if (typeof value !== 'string') throw new TypeError('MySet stores strings only');
  }

  has(value) {
    this.#checkValue(value);
    return Object.hasOwn(this.#items, value);
  }

  add(value) {
    if (this.has(value)) return false;
    this.#items[value] = true;
    this.#size++;
    return true;
  }

  addAll(values) {
    for (const value of values) this.add(value);
  }

  delete(value) {
    if (!this.has(value)) return false;
    delete this.#items[value];
    this.#size--;
    return true;
  }

  clear() {
    this.#items = Object.create(null);
    this.#size = 0;
  }

  get size() {
    return this.#size;
  }

  getSizeWithoutSizeProperty() {
    let count = 0;
    for (const key in this.#items) {
      if (Object.hasOwn(this.#items, key)) count++;
    }
    return count;
  }

  isEmpty() {
    return this.#size === 0;
  }

  values() {
    return Object.keys(this.#items);
  }

  union(otherSet) {
    const result = new MySet(this.values());
    result.addAll(otherSet.values());
    return result;
  }

  intersectionSimple(otherSet) {
    const result = new MySet();
    for (const value of this.values()) {
      if (otherSet.has(value)) result.add(value);
    }
    return result;
  }

  intersection(otherSet) {
    const result = new MySet();
    const [smaller, larger] = this.size <= otherSet.size
      ? [this, otherSet] : [otherSet, this];
    for (const value of smaller.values()) {
      if (larger.has(value)) result.add(value);
    }
    return result;
  }

  difference(otherSet) {
    const result = new MySet();
    for (const value of this.values()) {
      if (!otherSet.has(value)) result.add(value);
    }
    return result;
  }

  isSubsetOf(otherSet) {
    if (this.size > otherSet.size) return false;
    return this.values().every((value) => otherSet.has(value));
  }

  isSupersetOf(otherSet) {
    return otherSet.isSubsetOf(this);
  }
};

In [3]:
(() => {
  const values = new MySet();
  console.log('Initially empty:', values.isEmpty()); // true
  console.log('Add alpha:', values.add('alpha')); // true
  console.log('Duplicate alpha:', values.add('alpha')); // false
  values.addAll(['beta', 'gamma', 'beta']);
  console.log('Values:', values.values());
  console.log('Size getter:', values.size); // 3
  console.log('Size by counting:', values.getSizeWithoutSizeProperty()); // 3
  console.log('Has beta:', values.has('beta')); // true
  console.log('Delete beta:', values.delete('beta')); // true
  console.log('Delete missing beta:', values.delete('beta')); // false
  values.clear();
  console.log('After clear:', values.values(), values.size); // [] 0

  const special = new MySet(['toString', '__proto__', 'constructor']);
  console.log('Special names are ordinary members:', special.values());
  try {
    special.add(1);
  } catch (error) {
    console.log('Expected input error:', error.message);
  }
})();

Initially empty: true


Add alpha: true


Duplicate alpha: false


Values: [ 'alpha', 'beta', 'gamma' ]


Size getter: 3


...

Delete beta: true


Delete missing beta: false


After clear: [] 0


Special names are ordinary members: [ 'toString', '__proto__', 'constructor' ]


Expected input error: MySet stores strings only


## 3. Unique article tags

Adding the same tag twice leaves one member. Tag matching is case-sensitive: `'JavaScript'` and `'javascript'` are different strings.

In [4]:
(() => {
  const article = {
    title: 'The importance of data structures in programming',
    content: '...',
    tags: new MySet(),
  };
  article.tags.add('programming');
  article.tags.add('data structures');
  article.tags.add('algorithms');
  article.tags.add('programming');
  console.log(article.tags.size); // 3
  for (const tag of ['data structures', 'algorithms', 'programming', 'javascript']) {
    console.log(tag, article.tags.has(tag)); // true, true, true, false
  }
  console.log(article.tags.values());
  article.tags.delete('programming');
  article.tags.add('JavaScript');
  console.log(article.tags.values()); // ['data structures', 'algorithms', 'JavaScript']
})();

3


data structures true


algorithms true


programming true


javascript false


[ 'programming', 'data structures', 'algorithms' ]


[ 'data structures', 'algorithms', 'JavaScript' ]


## 4. Union, intersection, and difference

| Operation | Meaning | A = {1, 2, 3}, B = {3, 4, 5} |
| --- | --- | --- |
| A ∪ B | Members in either set | {1, 2, 3, 4, 5} |
| A ∩ B | Members in both sets | {3} |
| A − B | Members in A but not B | {1, 2} |
| B − A | Members in B but not A | {4, 5} |

Union and intersection are commutative; difference is directional. The custom class uses string versions of these numbers. Set operations also appear in SQL (`UNION`, `INTERSECT`, and `EXCEPT`); SQL can retain duplicates in other contexts, so query results are not always mathematical sets.

In [5]:
(() => {
  const a = new MySet(['1', '2', '3']);
  const b = new MySet(['3', '4', '5']);
  console.log('A union B:', a.union(b).values());
  console.log('A intersection B:', a.intersection(b).values());
  console.log('A minus B:', a.difference(b).values());
  console.log('B minus A:', b.difference(a).values());
  console.log('Unchanged inputs:', a.values(), b.values());
})();

A union B: [ '1', '2', '3', '4', '5' ]


A intersection B: [ '3' ]


A minus B: [ '1', '2' ]


B minus A: [ '4', '5' ]


Unchanged inputs: [ '1', '2', '3' ] [ '3', '4', '5' ]


### Iterating over the smaller set

The simple intersection scans every member of the first set. The optimized version scans the smaller input and checks membership in the larger one. For sizes n and m, this reduces membership checks from n to min(n, m).

In [6]:
(() => {
  const large = new MySet(['1', '2', '3', '4', '5', '6', '7']);
  const small = new MySet(['4', '6']);
  console.log('Simple:', large.intersectionSimple(small).values()); // ['4', '6']
  console.log('Optimized:', large.intersection(small).values());   // ['4', '6']
  console.log('Membership checks:', { simple: large.size, optimized: Math.min(large.size, small.size) });
})();

Simple: [ '4', '6' ]


Optimized: [ '4', '6' ]


Membership checks: { simple: 7, optimized: 2 }


## 5. Matching candidates with jobs

Intersect skills to find overlap, discard jobs with no shared skills, and sort by overlap count. The sort completes the ranking described in the draft. An overlap is a simple matching signal, not evidence that every job requirement is met.

In [7]:
var matchCandidateWithJobs = function matchCandidateWithJobs(candidate, jobPostings) {
  const matches = [];
  for (const job of jobPostings) {
    const matchingSkills = candidate.skills.intersection(job.skills);
    if (!matchingSkills.isEmpty()) {
      matches.push({ title: job.title, matchingSkills: matchingSkills.values() });
    }
  }
  return matches.sort((a, b) => b.matchingSkills.length - a.matchingSkills.length);
};

(() => {
  const job1Skills = new MySet();
  job1Skills.addAll(['JavaScript', 'Angular', 'Java', 'SQL']);
  const job2Skills = new MySet();
  job2Skills.addAll(['Python', 'Machine Learning', 'SQL', 'Statistics']);
  const jobPostings = [
    { title: 'Software Engineer', skills: job1Skills },
    { title: 'Data Scientist', skills: job2Skills },
  ];
  const candidate = {
    name: 'Loiane',
    skills: new MySet(['JavaScript', 'Angular', 'TypeScript', 'AWS']),
  };
  console.log(matchCandidateWithJobs(candidate, jobPostings));
  // [{ title: 'Software Engineer', matchingSkills: ['JavaScript', 'Angular'] }]
})();

[
  {
    title: 'Software Engineer',
    matchingSkills: [ 'JavaScript', 'Angular' ]
  }
]


## 6. Selecting subscribers with difference

Select subscribers interested in books and exclude those who already purchased. The `sendEmail` function below only prints a preview; it makes no network requests.

In [8]:
(() => {
  const allSubscribers = new MySet(['Aelin', 'Rowan', 'Xaden', 'Poppy', 'Violet']);
  const booksInterested = new MySet(['Aelin', 'Poppy', 'Violet']);
  const alreadyPurchasedBooks = new MySet(['Poppy']);
  const targetSubscribers = booksInterested
    .intersection(allSubscribers)
    .difference(alreadyPurchasedBooks);
  function sendEmail(subscriber, message) {
    console.log(`Email preview for ${subscriber}: ${message}`);
  }
  targetSubscribers.values().forEach((subscriber) => {
    sendEmail(subscriber, 'New books you will love!');
  });
  // Aelin and Violet
})();

Email preview for Aelin: New books you will love!


Email preview for Violet: New books you will love!


## 7. Combining results in an AI search pipeline

A retrieval-augmented generation (RAG) system may combine keyword and semantic retrieval. Here the search results are predefined document identifiers: no model, API key, or external search is needed. Sets organize these results; they do not perform retrieval or measure relevance.

Union collects all candidates, intersection identifies agreement, and difference identifies documents found by only one strategy. Agreement can inform ranking but does not guarantee correctness.

In [9]:
(() => {
  const keywordResults = new MySet(['doc-1', 'doc-2', 'doc-4']);
  const semanticResults = new MySet(['doc-2', 'doc-3', 'doc-4']);
  const candidateDocuments = keywordResults.union(semanticResults);
  const highConfidenceDocuments = keywordResults.intersection(semanticResults);
  const semanticOnlyDocuments = semanticResults.difference(keywordResults);
  console.log('All candidates:', candidateDocuments.values());
  // ['doc-1', 'doc-2', 'doc-4', 'doc-3']
  console.log('Found by both:', highConfidenceDocuments.values()); // ['doc-2', 'doc-4']
  console.log('Semantic only:', semanticOnlyDocuments.values()); // ['doc-3']
})();

All candidates: [ 'doc-1', 'doc-2', 'doc-4', 'doc-3' ]


Found by both: [ 'doc-2', 'doc-4' ]


Semantic only: [ 'doc-3' ]


## 8. Subsets, supersets, and recipe filtering

A ⊆ B means every member of A belongs to B; equivalently, B is a superset of A. Equal sets are subsets of each other. The empty set is a subset of every set.

A recipe can be made when its ingredients are a subset of the user's available ingredients. This section supplies both the `filterRecipes` function and the `userIngredients` data missing from the draft.

In [10]:
var filterRecipes = function filterRecipes(recipes, userIngredients) {
  const filteredRecipes = [];
  for (const recipe of recipes) {
    if (recipe.ingredients.isSubsetOf(userIngredients)) {
      filteredRecipes.push({ name: recipe.name });
    }
  }
  return filteredRecipes;
};

(() => {
  const chickenIngredients = new MySet(['chicken', 'tomato', 'onion', 'garlic', 'ginger', 'spices']);
  const spaghettiIngredients = new MySet(['spaghetti', 'eggs', 'bacon', 'parmesan', 'pepper']);
  const recipes = [
    { name: 'Chicken Tikka Masala', ingredients: chickenIngredients },
    { name: 'Spaghetti Carbonara', ingredients: spaghettiIngredients },
  ];
  const userIngredients = new MySet(['chicken', 'tomato', 'onion', 'garlic', 'ginger', 'spices', 'rice']);
  console.log(filterRecipes(recipes, userIngredients)); // [{ name: 'Chicken Tikka Masala' }]
  console.log('Pantry is a superset:', userIngredients.isSupersetOf(chickenIngredients)); // true
  console.log('Empty subset:', new MySet().isSubsetOf(userIngredients)); // true
})();

[ { name: 'Chicken Tikka Masala' } ]


Pantry is a superset: true


Empty subset: true


## 9. JavaScript's native Set

Native `Set` accepts an iterable, supports mixed types, returns itself from `add()` for chaining, and exposes `size`. Its `values()` and `keys()` return iterators, while `entries()` yields `[value, value]` pairs. Iteration follows insertion order. Objects are compared by identity; repeated `NaN` values count as one member, as do `0` and `-0`. See the [ECMAScript Set specification](https://tc39.es/ecma262/multipage/keyed-collections.html#sec-set-objects).

Unlike native `Set`, `MySet.values()` returns an array and its object-key ordering can move integer-like strings ahead of other keys. Neither implementation provides array-style positional indexing.

In [11]:
(() => {
  const article = {
    title: 'The importance of data structures in programming',
    content: '...',
    tags: new Set(),
  };
  article.tags.add('programming').add('data structures').add('algorithms').add('programming');
  console.log('Unique tags:', [...article.tags]);
  console.log('Size:', article.tags.size); // 3
  console.log('Has algorithms:', article.tags.has('algorithms')); // true
  article.tags.delete('programming');
  article.tags.add('JavaScript');
  console.log('Values iterator:', [...article.tags.values()]);
  console.log('Keys iterator:', [...article.tags.keys()]);
  console.log('Entries:', [...article.tags.entries()]);
  article.tags.forEach((value) => console.log('forEach:', value));
  for (const value of article.tags) console.log('for...of:', value);
  const fromArray = new Set(['programming', 'data structures', 'algorithms', 'programming']);
  console.log('Constructed from an array:', [...fromArray]);
  article.tags.clear();
  console.log('Empty after clear:', article.tags.size === 0); // true
})();

Unique tags: [ 'programming', 'data structures', 'algorithms' ]


Size: 3


Has algorithms: true


Values iterator: [ 'data structures', 'algorithms', 'JavaScript' ]


Keys iterator: [ 'data structures', 'algorithms', 'JavaScript' ]


...

for...of: data structures


for...of: algorithms


for...of: JavaScript


Constructed from an array: [ 'programming', 'data structures', 'algorithms' ]


Empty after clear: true


In [12]:
(() => {
  const object = { id: 1 };
  const mixed = new Set([1, '1', NaN, NaN, 0, -0, object, object, { id: 1 }]);
  console.log('Distinct mixed values:', mixed.size); // 6
  console.log('Original object found:', mixed.has(object)); // true
  console.log('New equal-looking object found:', mixed.has({ id: 1 })); // false
  const ordered = new Set(['10', '2', 'apple', '1']);
  console.log('Native insertion order:', [...ordered]); // ['10', '2', 'apple', '1']
  console.log('Custom object-key order:', new MySet(ordered).values()); // ['1', '2', '10', 'apple']
})();

Distinct mixed values: 6


Original object found: true


New equal-looking object found: false


Native insertion order: [ '10', '2', 'apple', '1' ]


Custom object-key order: [ '1', '2', '10', 'apple' ]


### Native set operations and compatibility

Modern runtimes provide native union, intersection, difference, symmetric difference, subset, superset, and disjointness methods. Older kernels may lack them. This cell detects support and uses local fallback functions so it also runs with the existing Node.js 20 kernel. No global prototypes are modified. See [ECMAScript's Set methods](https://tc39.es/ecma262/multipage/keyed-collections.html#sec-properties-of-the-set-prototype-object).

In [13]:
(() => {
  const a = new Set([1, 2, 3]);
  const b = new Set([3, 4, 5]);
  const fallbacks = {
    union: (x, y) => new Set([...x, ...y]),
    intersection: (x, y) => new Set([...x].filter((value) => y.has(value))),
    difference: (x, y) => new Set([...x].filter((value) => !y.has(value))),
    symmetricDifference: (x, y) => new Set([
      ...[...x].filter((value) => !y.has(value)),
      ...[...y].filter((value) => !x.has(value)),
    ]),
    isSubsetOf: (x, y) => [...x].every((value) => y.has(value)),
    isSupersetOf: (x, y) => [...y].every((value) => x.has(value)),
    isDisjointFrom: (x, y) => [...x].every((value) => !y.has(value)),
  };
  for (const [method, fallback] of Object.entries(fallbacks)) {
    const supported = typeof a[method] === 'function';
    const result = supported ? a[method](b) : fallback(a, b);
    console.log(method, supported ? '(native)' : '(fallback)', result instanceof Set ? [...result] : result);
  }
})();

union (fallback) [ 1, 2, 3, 4, 5 ]


intersection (fallback) [ 3 ]


difference (fallback) [ 1, 2 ]


symmetricDifference (fallback) [ 1, 2, 4, 5 ]


isSubsetOf (fallback) false


isSupersetOf (fallback) false


isDisjointFrom (fallback) false


## 10. Complexity and choosing a collection

For the custom class, assume typical average constant-time property lookup and bounded-length keys. Let n be this set's size, m the other set's size, and k an input iterable's length.

| Custom operation | Typical time |
| --- | --- |
| `add`, `delete`, `has` | O(1) average |
| `addAll` | O(k) |
| `size`, `isEmpty` | O(1) |
| `clear` | O(1) assignment, excluding later garbage collection |
| `values`, `getSizeWithoutSizeProperty` | O(n) |
| `union` | O(n + m) |
| `intersectionSimple` | O(n) |
| `intersection` | O(min(n, m)) |
| `difference`, `isSubsetOf` | O(n); subset size rejection is O(1) |
| `isSupersetOf` | O(m); size rejection is O(1) |

Stored members take O(n) space. `values()` creates an O(n) array, and set-producing operations allocate new results.

Native `Set` membership is commonly average O(1), but the specification requires average **sublinear access**, not a universal O(1) guarantee. Getting a native values iterator does not materialize an array; consuming all its elements takes O(n). See the [ECMAScript specification](https://tc39.es/ecma262/multipage/keyed-collections.html#sec-set-objects).

Choose a set for uniqueness and membership, and an array when duplicates matter or you need direct positional access. Native sets do preserve insertion order.

## 11. Exercise: remove duplicates from a sorted array

[LeetCode 26](https://leetcode.com/problems/remove-duplicates-from-sorted-array/description/) asks you to return the unique count k and write those unique values into the first k positions of the original array. Values after k are irrelevant; the array does not need to shrink.

The chapter's set-based approach takes O(n) typical time and O(k) extra space. Because the input is already sorted and native Set preserves insertion order, the unique prefix stays sorted. A set does not sort arbitrary input. The function below retains the chapter's name `removeDuplicates2`.

In [14]:
var removeDuplicates2 = function removeDuplicates2(nums) {
  const set = new Set(nums);
  const arr = Array.from(set);
  for (let i = 0; i < arr.length; i++) nums[i] = arr[i];
  return arr.length;
};

(() => {
  const nums = [0, 0, 1, 1, 1, 2, 2, 3, 3, 4];
  const originalReference = nums;
  const k = removeDuplicates2(nums);
  console.log('Unique count:', k); // 5
  console.log('Valid prefix:', nums.slice(0, k)); // [0, 1, 2, 3, 4]
  console.log('Same array object:', nums === originalReference); // true
  console.log('Full array (ignore the tail):', nums);
})();

Unique count: 5


Valid prefix: [ 0, 1, 2, 3, 4 ]


Same array object: true


Full array (ignore the tail): [
  0, 1, 2, 3, 4,
  2, 2, 3, 3, 4
]


### Optional challenge: O(1) extra space

Try solving the problem without a set before running this solution. For sorted input, each new value differs from the previous unique value. A write index marks where to place it. This version takes O(n) time and O(1) extra space.

In [15]:
var removeDuplicatesInPlace = function removeDuplicatesInPlace(nums) {
  let write = 0;
  for (let read = 0; read < nums.length; read++) {
    if (write === 0 || nums[read] !== nums[write - 1]) {
      nums[write] = nums[read];
      write++;
    }
  }
  return write;
};

(() => {
  const nums = [1, 1, 2];
  const k = removeDuplicatesInPlace(nums);
  console.log('Unique count:', k); // 2
  console.log('Valid prefix:', nums.slice(0, k)); // [1, 2]
})();

Unique count: 2


Valid prefix: [ 1, 2 ]


## 12. Runnable checks

These checks cover duplicate handling, special property names, string-only inputs, set algebra, unchanged inputs, application examples, and both duplicate-removal solutions. A failing assertion stops the cell with an error.

In [16]:
(() => {
  const assert = require('node:assert/strict');
  const sorted = (set) => set.values().sort();
  const values = new MySet();
  assert.equal(values.isEmpty(), true);
  for (const name of ['toString', '__proto__', 'constructor', 'hasOwnProperty', '']) {
    assert.equal(values.has(name), false);
    assert.equal(values.add(name), true);
    assert.equal(values.add(name), false);
    assert.equal(values.has(name), true);
  }
  assert.equal(values.size, 5);
  assert.equal(values.getSizeWithoutSizeProperty(), 5);
  assert.equal(values.delete('__proto__'), true);
  assert.equal(values.delete('__proto__'), false);
  assert.equal(values.size, 4);
  for (const value of [1, null, undefined, {}, Symbol('x')]) {
    assert.throws(() => values.add(value), TypeError);
  }
  values.clear();
  assert.equal(values.size, 0);
  assert.deepEqual(values.values(), []);
  assert.equal(values.has('toString'), false);
  assert.equal(values.add('__proto__'), true);

  // Compare every pair of subsets of a small universe with independent array expectations.
  const universe = ['a', 'b', 'c'];
  for (let left = 0; left < 8; left++) {
    for (let right = 0; right < 8; right++) {
      const x = universe.filter((_, i) => left & (1 << i));
      const y = universe.filter((_, i) => right & (1 << i));
      const a = new MySet(x), b = new MySet(y);
      assert.deepEqual(sorted(a.union(b)), [...new Set([...x, ...y])].sort());
      const common = x.filter((value) => y.includes(value));
      assert.deepEqual(sorted(a.intersectionSimple(b)), common);
      assert.deepEqual(sorted(a.intersection(b)), common);
      assert.deepEqual(sorted(a.difference(b)), x.filter((value) => !y.includes(value)));
      assert.equal(a.isSubsetOf(b), x.every((value) => y.includes(value)));
      assert.equal(a.isSupersetOf(b), y.every((value) => x.includes(value)));
      assert.deepEqual(a.values(), x);
      assert.deepEqual(b.values(), y);
    }
  }
  const candidate = { skills: new MySet(['JavaScript', 'SQL']) };
  assert.deepEqual(matchCandidateWithJobs(candidate, [
    { title: 'One match', skills: new MySet(['SQL']) },
    { title: 'No matches', skills: new MySet(['Python']) },
    { title: 'Two matches', skills: new MySet(['JavaScript', 'SQL']) },
  ]).map((job) => job.title), ['Two matches', 'One match']);
  assert.deepEqual(filterRecipes([
    { name: 'Available', ingredients: new MySet(['rice']) },
    { name: 'Missing', ingredients: new MySet(['eggs']) },
  ], new MySet(['rice', 'salt'])), [{ name: 'Available' }]);
  const keyword = new MySet(['doc-1', 'doc-2', 'doc-4']);
  const semantic = new MySet(['doc-2', 'doc-3', 'doc-4']);
  assert.deepEqual(sorted(keyword.union(semantic)), ['doc-1', 'doc-2', 'doc-3', 'doc-4']);
  assert.deepEqual(sorted(keyword.intersection(semantic)), ['doc-2', 'doc-4']);
  assert.deepEqual(semantic.difference(keyword).values(), ['doc-3']);
  assert.deepEqual(new MySet(['Aelin', 'Poppy', 'Violet'])
    .difference(new MySet(['Poppy'])).values(), ['Aelin', 'Violet']);

  for (const remove of [removeDuplicates2, removeDuplicatesInPlace]) {
    for (const input of [[], [1], [1, 1, 1], [1, 1, 2], [-2, -2, 0, 1, 1], [0, 1, 2]]) {
      const nums = [...input];
      const expected = input.filter((value, i) => i === 0 || value !== input[i - 1]);
      const k = remove(nums);
      assert.equal(k, expected.length);
      assert.deepEqual(nums.slice(0, k), expected);
    }
  }
  console.log('All Chapter 7 checks passed.');
})();

All Chapter 7 checks passed.


## Summary and experiments

`MySet` demonstrates uniqueness using object keys; native `Set` supports arbitrary values and insertion-order iteration. Union collects members, intersection finds overlap, difference excludes members, and subset checks establish containment.

Try adding new job postings, changing the available recipe ingredients, or combining different document results. Predict each result, edit the example cell, and rerun it. For the sorted-array exercise, compare the memory needs of the set-based and two-pointer solutions.